In [11]:
import os
import requests
import duckdb
from dotenv import load_dotenv

load_dotenv("../.env")
CLIENT_ID = os.environ["TWITCH_CLIENT_ID"]
CLIENT_SECRET = os.environ["TWITCH_CLIENT_SECRET"]

# 1. get an access token from Twitch
tok = requests.post("https://id.twitch.tv/oauth2/token", params={
    "client_id": CLIENT_ID,
    "client_secret": CLIENT_SECRET,
    "grant_type": "client_credentials",
}, timeout=30)
tok.raise_for_status()
token = tok.json()["access_token"]
print("Token received. Expires in", tok.json()["expires_in"] // 86400, "days")

# 2. test request to IGDB
headers = {"Client-ID": CLIENT_ID, "Authorization": f"Bearer {token}"}
body = """
fields name, first_release_date, aggregated_rating, aggregated_rating_count, platforms.name;
search "Celeste";
limit 3;
"""
r = requests.post("https://api.igdb.com/v4/games", headers=headers, data=body, timeout=30)
print(r.status_code)
print(r.json())

Token received. Expires in 63 days
200
[{'id': 26226, 'aggregated_rating': 91.3, 'aggregated_rating_count': 10, 'first_release_date': 1516838400, 'name': 'Celeste', 'platforms': [{'id': 170, 'name': 'Google Stadia'}, {'id': 48, 'name': 'PlayStation 4'}, {'id': 3, 'name': 'Linux'}, {'id': 6, 'name': 'PC (Microsoft Windows)'}, {'id': 14, 'name': 'Mac'}, {'id': 49, 'name': 'Xbox One'}, {'id': 130, 'name': 'Nintendo Switch'}]}, {'id': 76427, 'first_release_date': 1510531200, 'name': 'Celeste Witch', 'platforms': [{'id': 6, 'name': 'PC (Microsoft Windows)'}]}, {'id': 91484, 'first_release_date': 473299200, 'name': 'Marie Celeste', 'platforms': [{'id': 26, 'name': 'ZX Spectrum'}]}]


In [12]:
import pandas as pd

body = "fields id, name, generation, platform_family.name; limit 500;"
r = requests.post("https://api.igdb.com/v4/platforms", headers=headers, data=body, timeout=30)
plats = pd.DataFrame(r.json())

plats["family"] = plats["platform_family"].apply(lambda x: x["name"] if isinstance(x, dict) else None)
console_plats = plats[plats["family"].isin(["Nintendo", "PlayStation", "Xbox"])]
console_plats.sort_values(["family", "generation"])[["id", "name", "family", "generation"]]

,id,name,family,generation
97,307,Game & Watch,Nintendo,2.0
39,51,Family Computer Disk System,Nintendo,3.0
76,18,Nintendo Entertainment System,Nintendo,3.0
145,99,Family Computer,Nintendo,3.0
6,306,Satellaview,Nintendo,4.0
92,131,Super NES CD-ROM System,Nintendo,4.0
112,33,Game Boy,Nintendo,4.0
193,19,Super Nintendo Entertainment System,Nintendo,4.0
195,58,Super Famicom,Nintendo,4.0
26,87,Virtual Boy,Nintendo,5.0


In [13]:
plats[["id", "name", "family", "generation"]].to_csv("../data/processed/igdb_platforms.csv", index=False)

In [14]:
import json

games = pd.DataFrame(json.loads(open("../data/raw/igdb_console_games.json").read()))
plats = pd.read_csv("../data/processed/igdb_platforms.csv")

print(games.shape)
print(games.columns.tolist())

# check 1: is every game on at least one real console?
addons = {51, 306, 131, 416, 510, 47, 441}
console_ids = set(plats[plats["family"].isin(["Nintendo", "PlayStation", "Xbox"])]["id"]) - addons
games["on_console"] = games["platforms"].apply(lambda p: any(x in console_ids for x in p))
print("\nShare on a real console:", round(games["on_console"].mean(), 3))

# check 2: what types of entries are these? (main games, DLC, remasters...)
games["game_type_name"] = games["game_type"].apply(lambda x: x.get("type") if isinstance(x, dict) else None)
print("\n", games["game_type_name"].value_counts(dropna=False))

# check 3: how many critic reviews make up each rating?
print("\n", games["aggregated_rating_count"].describe().round(1))

(13186, 12)
['id', 'aggregated_rating', 'aggregated_rating_count', 'first_release_date', 'involved_companies', 'name', 'platforms', 'rating', 'rating_count', 'game_type', 'franchises', 'franchise']

Share on a real console: 1.0

 game_type_name
Main Game               10646
Bundle                    474
Port                      397
Expanded Game             363
DLC                       337
Remake                    284
Remaster                  280
Expansion                 181
Episode                   129
Standalone Expansion       66
Pack / Addon               14
Update                     10
Fork                        3
Season                      1
Mod                         1
Name: count, dtype: int64

 count    13186.0
mean         3.3
std          3.6
min          0.0
25%          1.0
50%          2.0
75%          4.0
max         52.0
Name: aggregated_rating_count, dtype: float64


In [15]:
main = games[games["game_type_name"] == "Main Game"].copy()

for t in [1, 2, 3, 5, 10]:
    print(f">= {t} critic reviews: {(main['aggregated_rating_count'] >= t).sum():,} games")

>= 1 critic reviews: 10,510 games
>= 2 critic reviews: 6,679 games
>= 3 critic reviews: 4,554 games
>= 5 critic reviews: 2,361 games
>= 10 critic reviews: 721 games


In [16]:
family = dict(zip(plats["id"], plats["family"]))
PC = {6, 14, 3}   # Windows, Mac, Linux (IDs seen in the Celeste test)

def exclusivity(platforms, allow_pc):
    families = {family[p] for p in platforms if p in console_ids}
    others = [p for p in platforms if p not in console_ids and p not in addons]
    if allow_pc:
        others = [p for p in others if p not in PC]
    if len(families) == 1 and not others:
        return f"{families.pop()} exclusive"
    return "Multiplatform"

main["excl_a"] = main["platforms"].apply(lambda p: exclusivity(p, allow_pc=True))    # A: PC allowed
main["excl_b"] = main["platforms"].apply(lambda p: exclusivity(p, allow_pc=False))   # B: strict

for t in [3, 5]:
    subset = main[main["aggregated_rating_count"] >= t]
    print(f"\n>= {t} critic reviews")
    print(pd.DataFrame({"A (PC allowed)": subset["excl_a"].value_counts(),
                        "B (strict)": subset["excl_b"].value_counts()}))


>= 3 critic reviews
                       A (PC allowed)  B (strict)
Multiplatform                    3166        3601
Nintendo exclusive                562         480
PlayStation exclusive             579         368
Xbox exclusive                    247         105

>= 5 critic reviews
                       A (PC allowed)  B (strict)
Multiplatform                    1737        1941
Nintendo exclusive                234         211
PlayStation exclusive             285         173
Xbox exclusive                    105          36


In [17]:
analysis = main[main["aggregated_rating_count"] >= 3][
    ["id", "name", "aggregated_rating", "aggregated_rating_count", "excl_a", "excl_b"]].copy()

def summarize(col, data_name="analysis"):
    return duckdb.sql(f"""
        SELECT
            {col}                                            AS group_name,
            COUNT(*)                                         AS games,
            ROUND(QUANTILE_CONT(aggregated_rating, 0.25), 1) AS p25,
            ROUND(MEDIAN(aggregated_rating), 1)              AS median,
            ROUND(QUANTILE_CONT(aggregated_rating, 0.75), 1) AS p75
        FROM {data_name}
        GROUP BY {col}
        ORDER BY median DESC
    """).df()

print("Definition A (PC allowed), 3+ critic reviews")
print(summarize("excl_a"))
print("\nDefinition B (strict), 3+ critic reviews")
print(summarize("excl_b"))

robust = analysis[analysis["aggregated_rating_count"] >= 5]
print("\nRobustness: Definition A, 5+ critic reviews")
print(summarize("excl_a", "robust"))

Definition A (PC allowed), 3+ critic reviews
              group_name  games   p25  median   p75
0          Multiplatform   3166  67.0    75.0  81.3
1     Nintendo exclusive    562  65.4    74.7  81.3
2  PlayStation exclusive    579  65.0    73.3  80.7
3         Xbox exclusive    247  63.3    73.3  81.3

Definition B (strict), 3+ critic reviews
              group_name  games   p25  median   p75
0     Nintendo exclusive    480  65.7    75.0  82.0
1          Multiplatform   3601  66.8    74.8  81.2
2  PlayStation exclusive    368  65.0    74.1  81.1
3         Xbox exclusive    105  55.0    68.7  77.3

Robustness: Definition A, 5+ critic reviews
              group_name  games   p25  median   p75
0     Nintendo exclusive    234  70.4    78.0  82.7
1          Multiplatform   1737  69.6    76.4  82.3
2         Xbox exclusive    105  64.7    75.6  83.7
3  PlayStation exclusive    285  67.4    75.6  82.1


In [18]:
first_party_names = ["Nintendo", "Sony", "Microsoft", "Xbox Game Studios"]

def publishers(companies):
    if not isinstance(companies, list):
        return []
    return [c["company"]["name"] for c in companies
            if c.get("publisher") and isinstance(c.get("company"), dict)]

analysis = analysis.merge(main[["id", "involved_companies"]], on="id")
analysis["publishers"] = analysis["involved_companies"].apply(publishers)
analysis["first_party"] = analysis["publishers"].apply(
    lambda pubs: any(fp in p for p in pubs for fp in first_party_names))

def party_group(row):
    if row["excl_a"] == "Multiplatform":
        return "Multiplatform"
    return row["excl_a"] + (" (first-party)" if row["first_party"] else " (third-party)")

analysis["party_group"] = analysis.apply(party_group, axis=1)
print(summarize("party_group"))

                            group_name  games   p25  median   p75
0     Nintendo exclusive (first-party)    270  71.4    78.7  84.9
1  PlayStation exclusive (first-party)    220  69.4    78.3  85.0
2         Xbox exclusive (first-party)    121  67.0    75.7  83.9
3                        Multiplatform   3166  67.0    75.0  81.3
4  PlayStation exclusive (third-party)    359  62.5    71.0  78.3
5     Nintendo exclusive (third-party)    292  61.8    70.2  77.7
6         Xbox exclusive (third-party)    126  58.4    69.3  77.3


**Finding (exclusives):** Overall, console exclusives receive similar critic ratings to multiplatform games (medians 73–75 under the main definition), and the ranking shifts across definitions and thresholds. Splitting exclusives by publisher reveals why: first-party exclusives (published by Nintendo, Sony or Microsoft) score above multiplatform games (medians 75.7–78.7 vs 75.0), while third-party exclusives score below (69.3–71.0). The pattern holds for all three console families; mixed together, the two groups cancel out. This is an association: first-party exclusives likely benefit from larger budgets and established franchises. First-party status is identified by publisher name and is approximate.